In [ ]:
# ============================================
# NOTEBOOK 1: Data Collection & Normalization (CẬP NHẬT - XỬ LÝ NGƯỢC SÁNG)
# ============================================

import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import shutil
import json
from tqdm import tqdm
from skimage import exposure, filters
from skimage.restoration import denoise_bilateral
import warnings
warnings.filterwarnings('ignore')

# Cấu hình
IMG_SIZE = 224  # hoặc 512
DATA_DIR = "/kaggle/input/datasets/grassknoted/asl-alphabet"
OUTPUT_DIR = "/kaggle/working/processed_data"

# Tạo thư mục output
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/images", exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/enhanced_images", exist_ok=True)  # Lưu ảnh đã cải thiện
os.makedirs(f"{OUTPUT_DIR}/reports", exist_ok=True)

print("=" * 50)
print("GIAI ĐOẠN 1: THU THẬP VÀ CHUẨN HÓA DỮ LIỆU")
print("=" * 50)

# ==================== CÁC HÀM XỬ LÝ NGƯỢC SÁNG ====================

def detect_backlight(image):
    """
    Phát hiện ảnh bị ngược sáng dựa trên histogram và độ tương phản
    
    Returns:
        is_backlit (bool): True nếu bị ngược sáng
        confidence (float): Độ tin cậy (0-1)
    """
    # Chuyển sang grayscale
    if len(image.shape) == 3:
        gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    else:
        gray = image
    
    # Tính histogram
    hist = cv2.calcHist([gray], [0], None, [256], [0, 256])
    hist = hist / hist.sum()
    
    # Đặc điểm 1: Pixel tối tập trung ở 2 đầu (ảnh ngược sáng)
    dark_pixels = hist[:50].sum()      # Pixel rất tối (0-50)
    bright_pixels = hist[200:].sum()   # Pixel rất sáng (200-255)
    mid_pixels = hist[50:200].sum()    # Pixel trung bình
    
    # Đặc điểm 2: Độ tương phản thấp
    contrast = gray.std() / 255.0
    
    # Đặc điểm 3: Kiểm tra bimodal distribution
    is_bimodal = (dark_pixels > 0.3 and bright_pixels > 0.2)
    
    # Đặc điểm 4: Phân bố pixel
    mean_intensity = gray.mean() / 255.0
    is_backlit = (mean_intensity < 0.4 and bright_pixels > 0.25) or is_bimodal
    
    # Tính độ tin cậy
    confidence = 0.0
    if is_bimodal:
        confidence = min(1.0, (dark_pixels + bright_pixels) / 0.8)
    elif mean_intensity < 0.3 and bright_pixels > 0.3:
        confidence = 0.8
    elif contrast < 0.15:
        confidence = 0.6
    
    return is_backlit, confidence, {
        'dark_ratio': float(dark_pixels),
        'bright_ratio': float(bright_pixels),
        'mid_ratio': float(mid_pixels),
        'contrast': float(contrast),
        'mean_intensity': float(mean_intensity)
    }


def correct_backlight(image, method='adaptive'):
    """
    Chỉnh sửa ảnh bị ngược sáng
    
    Args:
        image: Ảnh RGB
        method: 'clahe', 'gamma', 'adaptive', 'msrcr'
    
    Returns:
        corrected_image: Ảnh đã được cải thiện
    """
    if len(image.shape) == 3:
        img_lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)
        l_channel, a_channel, b_channel = cv2.split(img_lab)
    else:
        l_channel = image
        a_channel = b_channel = None
    
    if method == 'clahe':
        # CLAHE (Contrast Limited Adaptive Histogram Equalization)
        clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
        enhanced_l = clahe.apply(l_channel)
        
    elif method == 'gamma':
        # Gamma correction
        gamma = 1.5  # Tăng độ sáng cho vùng tối
        inv_gamma = 1.0 / gamma
        table = np.array([(i / 255.0) ** inv_gamma * 255 for i in range(256)]).astype("uint8")
        enhanced_l = cv2.LUT(l_channel, table)
        
    elif method == 'adaptive':
        # Kết hợp nhiều phương pháp
        # Bước 1: CLAHE
        clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8,8))
        temp = clahe.apply(l_channel)
        
        # Bước 2: Gamma correction
        gamma = 1.3
        inv_gamma = 1.0 / gamma
        table = np.array([(i / 255.0) ** inv_gamma * 255 for i in range(256)]).astype("uint8")
        enhanced_l = cv2.LUT(temp, table)
        
        # Bước 3: Denoise nhẹ
        enhanced_l = cv2.bilateralFilter(enhanced_l, 9, 75, 75)
        
    elif method == 'msrcr':
        # Multi-Scale Retinex with Color Restoration (đơn giản hóa)
        # Áp dụng Gaussian blur ở nhiều scale
        scales = [15, 31, 61]
        img_float = l_channel.astype(np.float32) / 255.0 + 1e-6
        
        retinex = np.zeros_like(img_float)
        for scale in scales:
            blur = cv2.GaussianBlur(img_float, (scale, scale), 0)
            retinex += np.log(img_float) - np.log(blur)
        
        retinex = retinex / len(scales)
        
        # Normalize
        retinex = (retinex - retinex.min()) / (retinex.max() - retinex.min())
        enhanced_l = (retinex * 255).astype(np.uint8)
    
    else:
        enhanced_l = l_channel
    
    # Ghép lại ảnh
    if a_channel is not None:
        enhanced_lab = cv2.merge([enhanced_l, a_channel, b_channel])
        corrected = cv2.cvtColor(enhanced_lab, cv2.COLOR_LAB2RGB)
    else:
        corrected = enhanced_l
    
    # Sharpening nhẹ để làm rõ đường nét
    kernel = np.array([[-1,-1,-1],
                       [-1, 9,-1],
                       [-1,-1,-1]])
    corrected = cv2.filter2D(corrected, -1, kernel)
    
    return corrected


def enhance_hand_edges(image):
    """
    Tăng cường đường viền bàn tay để trích xuất bone diagram tốt hơn
    """
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    
    # Bước 1: Bilateral filter giữ biên
    filtered = cv2.bilateralFilter(gray, 9, 75, 75)
    
    # Bước 2: Adaptive threshold
    binary = cv2.adaptiveThreshold(filtered, 255, 
                                   cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                   cv2.THRESH_BINARY_INV, 11, 2)
    
    # Bước 3: Morphological operations
    kernel = np.ones((3,3), np.uint8)
    morph = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)
    morph = cv2.morphologyEx(morph, cv2.MORPH_OPEN, kernel)
    
    # Bước 4: Edge detection
    edges = cv2.Canny(filtered, 30, 100)
    
    # Bước 5: Kết hợp
    enhanced = cv2.addWeighted(morph, 0.6, edges, 0.4, 0)
    
    return enhanced


# 1. Khám phá dataset
print("\n1. Khám phá cấu trúc dataset:")
train_path = f"{DATA_DIR}/asl_alphabet_train/asl_alphabet_train"
test_path = f"{DATA_DIR}/asl_alphabet_test/asl_alphabet_test"

# Lấy danh sách classes
classes = sorted([d for d in os.listdir(train_path) if os.path.isdir(os.path.join(train_path, d))])
print(f"Classes found: {len(classes)}")
print(f"First 10 classes: {classes[:10]}")

# Thống kê số lượng ảnh mỗi class
stats = []
for class_name in classes:
    class_dir = os.path.join(train_path, class_name)
    num_images = len([f for f in os.listdir(class_dir) if f.endswith(('.jpg', '.png', '.jpeg'))])
    stats.append({'class': class_name, 'count': num_images})

stats_df = pd.DataFrame(stats)
print(f"\nTổng số ảnh train: {stats_df['count'].sum()}")
print(f"Trung bình ảnh/class: {stats_df['count'].mean():.1f}")

# 2. Chuẩn hóa ảnh với xử lý ngược sáng
print("\n2. Đang chuẩn hóa ảnh và xử lý ngược sáng...")

processed_info = []
backlight_stats = []  # Thống kê về ảnh bị ngược sáng

for class_name in tqdm(classes[:29], desc="Processing classes"):
    class_output_dir = f"{OUTPUT_DIR}/images/{class_name}"
    class_enhanced_dir = f"{OUTPUT_DIR}/enhanced_images/{class_name}"
    os.makedirs(class_output_dir, exist_ok=True)
    os.makedirs(class_enhanced_dir, exist_ok=True)
    
    class_dir = os.path.join(train_path, class_name)
    images = [f for f in os.listdir(class_dir) if f.endswith(('.jpg', '.png', '.jpeg'))]
    
    for img_name in images[:1000]:
        img_path = os.path.join(class_dir, img_name)
        
        try:
            # Đọc ảnh
            img = cv2.imread(img_path)
            if img is None:
                continue
            
            # Chuyển RGB
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            
            # === PHÁT HIỆN NGƯỢC SÁNG ===
            is_backlit, confidence, metrics = detect_backlight(img_rgb)
            
            # Lưu thống kê
            backlight_stats.append({
                'class': class_name,
                'image': img_name,
                'is_backlit': is_backlit,
                'confidence': confidence,
                **metrics
            })
            
            # === XỬ LÝ NẾU BỊ NGƯỢC SÁNG ===
            if is_backlit and confidence > 0.5:
                # Chỉnh sửa ngược sáng
                img_corrected = correct_backlight(img_rgb, method='adaptive')
                img_to_save = img_corrected
                used_correction = True
            else:
                img_to_save = img_rgb
                used_correction = False
            
            # === TIỀN XỬ LÝ CHUẨN ===
            # Resize
            img_resized = cv2.resize(img_to_save, (IMG_SIZE, IMG_SIZE))
            
            # Chuẩn hóa pixel [0,1]
            img_normalized = img_resized.astype(np.float32) / 255.0
            
            # Lưu ảnh đã xử lý
            output_path = f"{class_output_dir}/{img_name}"
            cv2.imwrite(output_path, cv2.cvtColor((img_normalized * 255).astype(np.uint8), cv2.COLOR_RGB2BGR))
            
            # Lưu ảnh enhanced riêng (cho bone diagram)
            if used_correction:
                # Tăng cường thêm cho bone diagram
                enhanced = enhance_hand_edges(img_corrected)
                enhanced_resized = cv2.resize(enhanced, (IMG_SIZE, IMG_SIZE))
                enhanced_path = f"{class_enhanced_dir}/{img_name}"
                cv2.imwrite(enhanced_path, enhanced_resized)
            
            processed_info.append({
                'class': class_name,
                'image': img_name,
                'original_shape': img.shape,
                'processed_shape': img_resized.shape,
                'was_backlit': is_backlit,
                'corrected': used_correction,
                'confidence': confidence
            })
            
        except Exception as e:
            print(f"Error processing {img_path}: {e}")

print(f"\nĐã xử lý xong: {len(processed_info)} ảnh")

# === THỐNG KÊ VỀ ẢNH NGƯỢC SÁNG ===
backlight_df = pd.DataFrame(backlight_stats)
print("\n" + "="*50)
print("THỐNG KÊ ẢNH NGƯỢC SÁNG:")
print("="*50)
print(f"Tổng số ảnh kiểm tra: {len(backlight_df)}")
print(f"Số ảnh bị ngược sáng: {backlight_df['is_backlit'].sum()}")
print(f"Tỷ lệ ngược sáng: {backlight_df['is_backlit'].mean()*100:.2f}%")

# Thống kê theo từng class
backlit_by_class = backlight_df.groupby('class')['is_backlit'].agg(['count', 'sum', 'mean'])
backlit_by_class.columns = ['total', 'backlit_count', 'backlit_ratio']
print("\nTop 5 class có nhiều ảnh ngược sáng nhất:")
print(backlit_by_class.sort_values('backlit_ratio', ascending=False).head(10))

# Lưu báo cáo
backlight_df.to_csv(f"{OUTPUT_DIR}/reports/backlight_detection_report.csv", index=False)

# Vẽ biểu đồ phân bố ngược sáng
plt.figure(figsize=(15, 6))
classes_for_plot = backlit_by_class.index[:29]
ratios = backlit_by_class['backlit_ratio'][:29].values
plt.bar(classes_for_plot, ratios)
plt.xticks(rotation=90)
plt.ylabel('Tỷ lệ ảnh ngược sáng')
plt.title('Phân bố ảnh ngược sáng theo từng class (A-Z)')
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/reports/backlight_distribution.png")
plt.show()

# 3. Lưu metadata
metadata = {
    'img_size': IMG_SIZE,
    'classes': classes[:29],
    'num_classes': len(classes[:29]),
    'total_images': len(processed_info),
    'backlit_images': int(backlight_df['is_backlit'].sum()),
    'backlit_ratio': float(backlight_df['is_backlit'].mean()),
    'class_mapping': {i: cls for i, cls in enumerate(classes[:29])},
    'inverse_class_mapping': {cls: i for i, cls in enumerate(classes[:29])},
    'preprocessing': {
        'backlight_correction': 'adaptive',
        'edge_enhancement': True,
        'normalization': 'minmax_0_1'
    }
}

with open(f"{OUTPUT_DIR}/metadata.json", 'w') as f:
    json.dump(metadata, f, indent=2)

# Lưu thông tin xử lý
processed_df = pd.DataFrame(processed_info)
processed_df.to_csv(f"{OUTPUT_DIR}/processing_info.csv", index=False)

print(f"\nMetadata lưu tại: {OUTPUT_DIR}/metadata.json")

# 4. Hiển thị so sánh trước/sau xử lý ngược sáng
print("\n3. Hiển thị so sánh trước và sau xử lý ngược sáng:")

# Tìm một số ảnh bị ngược sáng để demo
backlit_samples = backlight_df[backlight_df['is_backlit'] == True].head(3)  # Lấy 3 ảnh thôi

if len(backlit_samples) > 0:
    # Mỗi ảnh sẽ hiển thị 2 cột (before/after) => 3 ảnh = 3 hàng x 2 cột
    fig, axes = plt.subplots(len(backlit_samples), 2, figsize=(12, 4*len(backlit_samples)))
    
    # Nếu chỉ có 1 ảnh, axes là 1D, cần xử lý riêng
    if len(backlit_samples) == 1:
        axes = axes.reshape(1, -1)
    
    for idx, (_, row) in enumerate(backlit_samples.iterrows()):
        # Đọc ảnh gốc
        original_path = f"{train_path}/{row['class']}/{row['image']}"
        original = cv2.imread(original_path)
        original_rgb = cv2.cvtColor(original, cv2.COLOR_BGR2RGB)
        
        # Đọc ảnh đã xử lý
        processed_path = f"{OUTPUT_DIR}/images/{row['class']}/{row['image']}"
        processed = cv2.imread(processed_path)
        processed_rgb = cv2.cvtColor(processed, cv2.COLOR_BGR2RGB)
        
        # Hiển thị BEFORE (cột 0)
        axes[idx, 0].imshow(original_rgb)
        axes[idx, 0].set_title(f"BEFORE: {row['class']} (conf: {row['confidence']:.2f})", fontsize=10)
        axes[idx, 0].axis('off')
        
        # Hiển thị AFTER (cột 1)
        axes[idx, 1].imshow(processed_rgb)
        axes[idx, 1].set_title(f"AFTER: {row['class']} (corrected)", fontsize=10)
        axes[idx, 1].axis('off')
    
    plt.suptitle("So sánh ảnh trước và sau khi xử lý ngược sáng", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/reports/backlight_correction_demo.png", bbox_inches='tight', dpi=150)
    plt.show()
else:
    print("Không tìm thấy ảnh ngược sáng trong mẫu để demo")

# 5. Hiển thị mẫu ảnh sau chuẩn hóa
print("\n4. Hiển thị mẫu ảnh sau chuẩn hóa:")
fig, axes = plt.subplots(3, 9, figsize=(15, 6))
axes = axes.ravel()

for i, class_name in enumerate(classes[:27]):
    sample_path = f"{OUTPUT_DIR}/images/{class_name}"
    if os.path.exists(sample_path):
        samples = os.listdir(sample_path)
        if samples:
            img = cv2.imread(f"{sample_path}/{samples[0]}")
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            axes[i].imshow(img_rgb)
            axes[i].set_title(class_name)
            axes[i].axis('off')

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/sample_images.png")
plt.show()

# 6. Tạo dataset cho Kaggle
print("\n5. Đóng gói dataset cho Kaggle...")

# Tạo file zip
shutil.make_archive(f"{OUTPUT_DIR}/asl_processed", 'zip', OUTPUT_DIR, "images")
shutil.make_archive(f"{OUTPUT_DIR}/asl_enhanced", 'zip', OUTPUT_DIR, "enhanced_images")
shutil.make_archive(f"{OUTPUT_DIR}/asl_metadata", 'zip', OUTPUT_DIR, "metadata.json")
shutil.make_archive(f"{OUTPUT_DIR}/asl_reports", 'zip', OUTPUT_DIR, "reports")

print(f"\n✅ HOÀN THÀNH GIAI ĐOẠN 1!")
print(f"Output saved to: {OUTPUT_DIR}")
print("\n📊 KẾT QUẢ XỬ LÝ:")
print(f"   - Tổng số ảnh: {len(processed_info)}")
print(f"   - Ảnh ngược sáng đã xử lý: {backlight_df['is_backlit'].sum()}")
print(f"   - Tỷ lệ ngược sáng: {backlight_df['is_backlit'].mean()*100:.2f}%")
print("\n📌 Lưu ý: Để dùng cho notebook tiếp theo:")
print("   1. Save version notebook này")
print("   2. Add /kaggle/working/processed_data làm dataset output")
print("   3. Ở notebook 2, sử dụng enhanced_images cho bone diagram extraction")